## Récupération des données

In [1]:
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
import seaborn as sns

# Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# Model
from sklearn.ensemble import RandomForestRegressor

# Option pour afficher toutes les colonnes
pd.set_option('display.max_columns', None)

## Récupération des données

In [2]:
print("Chargement des données en cours...")
start_time = time.time()

X_train = pd.read_csv('X_train.csv')
X_test = pd.read_csv('X_test.csv')

# Pour y, on s'assure d'avoir un vecteur plat (1D)
y_train = pd.read_csv('y_train.csv').values.ravel()
y_test = pd.read_csv('y_test.csv').values.ravel()

load_time = time.time() - start_time
print(f"Données chargées en {load_time:.2f} secondes.")
print(f"Dimensions Train : {X_train.shape} | Dimensions Test : {X_test.shape}")

Chargement des données en cours...
Données chargées en 8.92 secondes.
Dimensions Train : (3712444, 13) | Dimensions Test : (928111, 13)


## Entrainement

In [4]:
!pip install lightgbm

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   --------------------- ------------------ 0.8/1.5 MB 11.7 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 12.1 MB/s  0:00:00



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import lightgbm as lgb
import time

# 1. Préparation des données spécifiques à LightGBM
# Il peut gérer les catégories automatiquement, mais avec tes X_train déjà encodés, c'est parfait.
train_data = lgb.Dataset(X_train, label=y_train)

# 2. Paramètres
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'num_leaves': 31,          # Équivalent à une profondeur modérée
    'learning_rate': 0.05,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'n_jobs': -1,
    'random_state': 42,
    'verbose': -1
}

print("Début de l'entraînement LightGBM...")
start_time = time.time()
lgbm_model = lgb.train(params, train_data, num_boost_round=1000)
print(f"Entraînement terminé en {(time.time() - start_time)/60:.2f} minutes.")

# 3. Prédictions
y_pred_lgbm = lgbm_model.predict(X_test)
r2_lgbm = r2_score(y_test, y_pred_lgbm)
mae_lgbm = mean_absolute_error(y_test, y_pred_lgbm)

print(f"--- RÉSULTATS LIGHTGBM ---")
print(f"R² Score : {r2_lgbm:.4f}")
print(f"MAE      : {mae_lgbm:.2f} €")

Début de l'entraînement LightGBM...
Entraînement terminé en 2.45 minutes.
--- RÉSULTATS LIGHTGBM ---
R² Score : 0.7400
MAE      : 61930.83 €
